<a href="https://colab.research.google.com/github/seth-coder-cmyk/DH-Project/blob/main/JSD_explained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Jensen-Shannon Divergence

In this script we're working on an example to mathematically determine the similarity between two topically aligned Wikipedia articles in alemmanic and standard German. <br>
To do this, we represent the weighted tags of each article by a vector and compute the so-called <i>Jensen-Shannon Divergence</i> between them.

#Articles
In this example we look at the Wikipedia articles <i>Prähistorische Pfahlbauten um die Alphen</i> in alemmanic German (article_id: 57327) and standard German (article_id: 6307013).<br>
Their weighted tags are as follows:<br>
<br>
Alemannic:<br>
["classification.prediction.articlecountry/Switzerland|1000",<br> "classification.prediction.articlecountry/Slovenia|1000",<br>  "classification.prediction.articlecountry/Germany|1000",<br>  "classification.prediction.articlecountry/Austria|1000",<br>  "classification.prediction.articlecountry/France|1000",<br>  "classification.prediction.articlecountry/Italy|1000",<br>  "classification.prediction.articletopic/Geography.Regions.Europe.Europe*|781",<br>  "classification.prediction.articletopic/Geography.Regions.Europe.Western_Europe|988"] <br>
<br>
Standard:<br>
["classification.prediction.articlecountry/Switzerland|771",<br>  "classification.prediction.articlecountry/Austria|666",<br>  "classification.prediction.articlecountry/France|1000",<br>  "classification.prediction.articlecountry/Slovenia|666",<br>  "classification.prediction.articlecountry/Germany|333",<br>  "classification.prediction.articlecountry/Italy|1000",<br>  "classification.prediction.articletopic/Geography.Regions.Europe.Europe*|990",<br>  "classification.prediction.articletopic/Geography.Geographical|705",<br>  "classification.prediction.articletopic/Geography.Regions.Europe.Western_Europe|963"]

#Represent as Vectors
To work with these weighted tags in a more meaninful and mathematical way we need to first represent them as vectors (a = alemannic, b = standard). For this it is important that we first understand a few key things:<br>


*   The weight of the tags is determined on a scale of 0 to 1000 from least to most fitting for the category tag
*   The Jensen-Shannon Divergence works on a basis, that each set of tags accumulated make up 100% of the tags used
*   The vectors / lists of tags we want to compare need to be of the same dimension / length
<br>
That being said, we need to do a few things first.


1.   Make a global list <i>V</i> containing all classification tags
2.   Summarize all weights of the tags. Tags that are only contained in one article but are missing in the other will be given the weight of 0 for the article it's missing in. In this example, <i>geographical</i> is missing from the alemannic article. Instead of leaving it away, it will be given the weight 0
3.   Normalize the weights by dividing the tag frequency <i>i</i> by the total tag frequency

This leaves us with the following results:<br>
V = {Switzerland, Austria, France, Slovenia, Germany, Italy, Europe, Geographical, Western_Europe}<br>
A = {1000, 1000, 1000, 1000, 1000, 1000, 781, 0, 988}<br>
B = {771, 666, 1000, 666, 333, 1000, 990, 705, 963}




#Normalizing the Vectors
Apparently, the sum of our weights is <b>not</b> 100%, but a = 7769 and b = 7094. So we divide each valuve within our lists by the sum of their respective weights.<br>
Resulting in the following vectors:<br>
a = {0.129, 0.129, 0.129, 0.129, 0.129, 0.129, 0.1, 0, 0.127}<br>
b = {0.109, 0.094, 0.141, 0.094, 0.047, 0.141, 0.14, 0.099, 0.136}

#Computing the JSD
In this example, we're looking at <b>how</b> the JSD works mathematically before we're going to import it from the library for future implementation. <br>

The basis of the Jensen-Shanon-Divergence is the so-called Kullback-Leibler-Divergence, KL-Divergence for short.<br>
The KL-divergence is a metric for the divergence between two probability distributions using the logarith. So let's start with this:

In [1]:
import numpy as np

def kl_divergence(p, q, esp=1e-12):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)

    p = p + esp
    q = q + esp

    return np.sum(p * np.log(p / q))

#Extending the KL-Divergence to get to the JSD
Since the KL-divergence on its own is an asymmetric and limited measure for probability distributions we use the so-called Jensen-Shannon-Divergence. It is a symmetric metric and, using the logarithmic basis 2, its results will move on a scale between 0 and 1. <br>
Let's implement this next:

In [2]:
def jensen_shannon_divergence(p, q, base=np.e):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)

    #normalize to probability distribution
    p = p / np.sum(p)
    q = q / np.sum(q)

    m = 0.5 * (p + q)

    js = 0.5 * kl_divergence(p, m) + 0.5 * kl_divergence(q, m)

    if base == 2:
        return js / np.log(2)
    return js

#Application
Let us now apply this to our examples of our vectors a and b:<br>
(Note: don't be confused by the usage of p and q within the formula. These are just variable names. We will use the values of a and b in p and q respectively.)

In [3]:
p = [0.129, 0.129, 0.129, 0.129, 0.129, 0.129, 0.1, 0, 0.127]
q = [0.109, 0.094, 0.141, 0.094, 0.047, 0.141, 0.14, 0.099, 0.136]

In [4]:
print(jensen_shannon_divergence(p, q))

0.049389050837748194


#Result and Interpretation
The result of the JSD is approx. 0.049.

To interpret this we need to remember: The Jensen-Shannon-<b>Divergence</b> measures, how far two topic structures diverge from each other. This means: <br>
JSD = 0 - the topic structure is identical
JSD > 0 - increasingly different topical emphasis, with 1 meaning, that they are absolutely different from each other. <br>

For our result that means, that both the alemannic and the standard German article used in this example are very simmilar in its emphasis given the weighted tag distribution.